# ME 498/598 Challenge Problem 3

<div style="font-size: small;">
License Terms:
These Python demos are licensed under a <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International License</a>. They are intended for educational use only in ME 498/598: Applied AI for Engineering and Design at the University of Illinois Urbana-Champaign. You may not share or distribute them publicly, use them for commercial purposes, or provide them to industry or other entities without permission from the instructor (regnwttr@illinois.edu).
</div>

In [ ]:
import os
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import cp3_public as P

# Overview

You're an engineer at a rapid-prototyping startup that designs and cuts custom **plate brackets**. A customer tells you where their bracket is bolted down, where the load is applied and in which direction, and the most the bracket may weigh. Your company designs the bracket and cuts it from an 80 mm × 80 mm metal plate.

The company is launching **live design software**. Every customer should receive **3 concept designs** to choose from **within the hour**, and at peak demand the software must keep up with **1,000 customers per hour**. That's 3,000 designs per hour.

There's a catch: every design has to be **manufacturable on the company's equipment** and has to hold up in service. Nobody ever wrote these requirements down. What you have instead is the shop's record: thousands of previous designs, each labeled by your colleagues who run the machines as **accepted** (valid) or **rejected**. They didn't record why.

The CEO wants to see that the software works. Every hour, the CEO runs it on a fresh batch of 1,000 customer orders and quietly scores 100 of them. For each of those orders, the CEO has a target stiffness in mind. Knowing the CEO, she may have unrealistic expectations. That test is the **leaderboard**.

This notebook walks you through the problem, the data, the scoring and the submission format, and builds a simple baseline. You saw the design problem, SIMP and a conditional diffusion model in Demo 8; everything from that demo is fair game here.

# The Design Problem

This is the same bracket problem as in Demo 8:

- **Design:** an 80 × 80 binary image, where 1 is material and 0 is empty space. One pixel is 1 mm.
- **Condition** (one customer order): 8 numbers, `P.COND_COLS`:
  - `load_x, load_y`: the centre of the load disc (radius 3 mm), in mm, with x to the right and y *down* (like the image's rows and columns).
  - `force_x, force_y`: the direction of the load, a unit vector in the same frame.
  - `vf_cap`: the customer's weight limit, as the largest share of the plate the bracket may cover (between 0.2 and 0.5).
  - `sup_1, sup_2, sup_3`: the centres of the three 8 mm clamped supports, as distances along the plate's boundary, clockwise from the top-left corner.
- **Performance:** the **compliance** $C$, the work done by the load. Lower is stiffer. `P.compliance(design, cond)` computes it with finite element analysis (FEA). The load is shared equally by the nodes of the load disc that touch material.

`cp3_public.py` has everything from Demo 8: `P.compliance`, `P.compliance_gradient`, `P.simp`, `P.condition_image`, `P.plot_design(s)`, and more. Have a look at the file.

# The Data

The shop's record has 12,680 designs. Each comes with its condition, its compliance and its validity label. These are the same designs you trained on in Demo 8, now with the labels.

In [ ]:
X, cond, C, valid = P.load_designs('datasets/cp3_brackets.npz', labels=True)
print(f'{len(X)} designs, {valid.sum()} accepted ({valid.mean():.1%})')
print('designs', X.shape, ' conditions', cond.shape, ' compliance', C.shape, ' valid', valid.shape)

Here are some accepted and some rejected designs:

In [ ]:
iv, ii = np.where(valid)[0][:6], np.where(~valid)[0][:6]
P.plot_designs(np.concatenate([X[iv], X[ii]]), np.concatenate([cond[iv], cond[ii]]),
               [f'accepted, C = {C[i]:.3g}' for i in iv] + [f'rejected, C = {C[i]:.3g}' for i in ii])
plt.show()

Can you tell what the machinists look for? Partly, maybe. That's the challenge: you have to learn the requirements from the labels.

How does the acceptance rate depend on the volume cap, and on the compliance?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.linspace(0.2, 0.5, 7)
k = np.digitize(cond[:, 4], bins) - 1
axs[0].bar((bins[:-1] + bins[1:]) / 2, [valid[k == b].mean() for b in range(6)], width=0.045)
axs[0].set_xlabel('vf_cap'); axs[0].set_ylabel('share accepted')

logC = np.log10(C)
cbins = np.linspace(0, 6, 13)
kc = np.digitize(logC, cbins) - 1
share = [valid[kc == b].mean() if (kc == b).sum() > 20 else np.nan for b in range(12)]
axs[1].bar((cbins[:-1] + cbins[1:]) / 2, share, width=0.45)
axs[1].set_xlabel('log10(compliance)'); axs[1].set_ylabel('share accepted')
plt.tight_layout(); plt.show()

A few things to know about the data:

- **Accepted is not the same as good.** Many accepted designs are floppy, and a rejected design can be very stiff. The score below rewards designs that are both.
- **The manufactured part is not exactly your design.** The volume cap applies to the part as it comes off the machine, which can contain more material than the design you submit.
- **Compliance above about $10^4$ means the load path is broken**: the material doesn't properly connect the load to the supports. The FEA models empty space as a very soft material, so such designs get a huge but finite compliance.

# Scoring

For each scored condition, the CEO compares the compliance of your best concept with the target for that order:

$$\text{score} = \max_{\text{your 3 concepts}} \begin{cases} \min\left(1,\ C_\text{target} / C\right) & \text{if the concept is accepted} \\ 0 & \text{if it is rejected} \end{cases}$$

- $C$ is the compliance of your concept *as manufactured*.
- $C_\text{target}$ is the target compliance for that order. It depends on the order, the CEO knows it, and you don't. Scaling by it puts easy and hard orders on the same footing.
- A score of 1 means your best concept is accepted and at least as stiff as the target. A rejected concept scores 0, no matter how stiff it is. Between those, a stiffer accepted concept always scores higher. Remember the CEO's expectations: scores well below 1 are normal.
- Only your **best** concept counts, so three different concepts are worth much more than three copies of one. If one concept gets rejected, another can still save the order.

The round score is the mean over the 100 scored conditions.

Since the targets are hidden, you can't compute your score yourself. But `P.compliance` is a good guide while you develop your method: among accepted designs, lower compliance always scores at least as well. Keep in mind that it measures the design as submitted, not as manufactured.

# Rounds and the Leaderboard

Every hour, a new **round** of 1,000 conditions opens. You submit 3 concepts for each of its 1,000 conditions; 100 of them, hidden from you and the same for every team, are scored.

- A submission counts for the **current or the previous** round.
- Each team can upload at most **5 submissions per rolling 24 hours**.
- Your leaderboard score is the score of your **most recent** scored submission.

The leaderboard is at **https://corelab.mechse.illinois.edu/CP3/**. Download the current round there, and submit with the same private submitter ID you used for CP2. The page lists the full rules.

The leaderboard page has a **Download** link for the current round, a file named like `r0123.npz`. Save it in the `rounds/` folder of this repository, next to `r0000.npz`, the practice round that comes with it.

`P.load_round()` with no argument loads the **newest** round in `rounds/` (the highest id), so after each download it picks up the new round by itself. It prints which round it loaded, so check that it's the one you just downloaded. A round file holds its id and its 1,000 conditions:

In [ ]:
round_id, round_conds = P.load_round()                      # newest round in rounds/
# round_id, round_conds = P.load_round('rounds/r0000.npz')  # or a specific round file
print(round_id, round_conds.shape)
P.plot_designs(np.zeros((6, 80, 80)), round_conds[:6], [f'condition {k}' for k in range(6)])
plt.show()

# Starter Code: Retrieval

As a first baseline, we don't design anything new. For each condition in the round, we look up the 3 **accepted** designs in the data whose conditions are most similar, and submit those.

To compare conditions, we describe each one by its load position and direction, its volume cap, and a coarse 8 × 8 map of where its supports are. The weights in front of each part are a rough guess; feel free to change them.

In [ ]:
def condition_features(conds):
    feats = []
    for c in conds:
        supports = P.condition_image(c)[0]                             # 1 on the elements touching a support
        coarse = supports.reshape(8, 10, 8, 10).mean(axis=(1, 3)).ravel()   # 80 x 80 -> 8 x 8
        feats.append(np.r_[c[0] / 80, c[1] / 80, 0.5 * c[2], 0.5 * c[3], 2 * c[4], 3 * coarse])
    return np.array(feats)

X_ok, cond_ok = X[valid], cond[valid]                    # only accepted designs
f_data, f_round = condition_features(cond_ok), condition_features(round_conds)

dist = ((f_round[:, None, :] - f_data[None, :, :]) ** 2).sum(axis=-1)   # [1000 x n_accepted] squared distances
nearest = np.argsort(dist, axis=1)[:, :P.N_CONCEPTS]                    # 3 nearest accepted designs per condition
submission = X_ok[nearest]                                              # [1000 x 3 x 80 x 80]
print(submission.shape)

Let's look at the concepts for a few conditions, with their compliance under the *new* condition:

In [ ]:
designs, conds, titles = [], [], []
for k in range(4):
    for j in range(P.N_CONCEPTS):
        designs.append(submission[k, j]); conds.append(round_conds[k])
        titles.append(f'condition {k}, concept {j + 1}: C = {P.compliance(submission[k, j], round_conds[k]):.3g}')
P.plot_designs(designs, conds, titles, ncols=6)
plt.show()

The retrieved designs were made for *other* conditions, so they often miss a support or the load, and when they do connect, they are rarely as stiff as they could be. We can check the whole submission with the FEA:

In [ ]:
n_check = 50                                              # FEA takes a moment, so we check the first 50
C_sub, vf_sub = P.evaluate(submission[:n_check].reshape(-1, 80, 80), np.repeat(round_conds[:n_check], P.N_CONCEPTS, axis=0))
C_best = C_sub.reshape(n_check, P.N_CONCEPTS).min(axis=1)
print(f'Conditions where at least one concept connects the load (C < 1e4): {(C_best < 1e4).mean():.0%}')
print(f'Median compliance of the stiffest concept: {np.median(C_best):.3g}')

This baseline scores about **0.21**. Submitting only the single nearest design (three times) scores about 0.10, which shows how much the second and third concepts are worth.

There's a lot of room for improvement. Some directions:

- **Generate** designs for each condition instead of retrieving them, for example with the conditional diffusion model from Demo 8, trained on accepted designs.
- **Learn the shop's requirements**: train a classifier that predicts acceptance, and use it to filter, rank or guide your designs.
- **Optimize** designs with the compliance gradient (`P.compliance_gradient`), as in the guidance part of Demo 8, while staying within the volume cap and the shop's requirements.
- **Diversify** your 3 concepts so that they don't all fail the same way.
- **Stay fast**: your method has to deliver 3,000 designs within the hour.

# Preparing a Submission

A submission is one file with the round id and a `[1000 x 3 x 80 x 80]` array: 3 concepts for each condition, in the round's order. `P.save_submission` checks the shape and packs the designs into a small file (well under 1 MB).

In [ ]:
os.makedirs('submissions', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
path = f'submissions/{round_id}_{timestamp}.npz'
P.save_submission(path, submission, round_id)

rid, back = P.load_submission(path)                       # check that it reads back
assert rid == round_id and back.shape == (1000, P.N_CONCEPTS, 80, 80)
print(f'saved {path} ({os.path.getsize(path) / 1e6:.1f} MB)')

To submit for real:

1. Download the current round from the [leaderboard page](https://corelab.mechse.illinois.edu/CP3/) into `rounds/` and load it with `P.load_round()`. Check the round id it prints.
2. Run your method and save the submission as above.
3. Upload the `.npz` file on the leaderboard page with your submitter ID. It's scored in a queue, usually within a few minutes; look it up under "My submissions".

A submission only counts for the current or the previous round, so download, run and upload before the end of the following hour. Uploads for `r0000` are not accepted.